# Imports

In [0]:
# init
from pyspark.sql import Window
from pyspark.sql import functions as F
from pyspark.sql.functions import col, when, upper, lower
from pyspark.sql.types import StringType, IntegerType, DateType

# Read file from the bronze_schema

In [0]:
df = spark.table("workspace.bronze_schema.crm_sales_details")
df.display()

# Explore the data

In [0]:
df.describe().display()
"""
    From the count, I can see that some columns may have null values
"""

In [0]:
df.printSchema()

"""
    Dates are of IntegerType and should be converted to DataType
"""

## Check for white spaces in StringType column

In [0]:
for field in df.schema.fields:
    column = field.name
    column_type = field.dataType

    if isinstance(column_type, StringType):
        white_space_df = df.select("*").where(F.trim(col(column)) != col(column))
        white_space_count = white_space_df.count()


        if white_space_count:
            print(f"White space count {white_space_count} detected in {column}")
            white_space_df.display()


"""
    No white space detected in the StringType columns
"""

## Check for null values

In [0]:
for column in df.columns:
    null_df = df.select("*").where(col(column).isNull())
    null_count = null_df.count()

    if null_count:
        print(f"Null values of count {null_count} detected in Column: {column} ")
        null_df.display()

"""
    Null values detected in 2 columns sls_sales and sls_price
    Price and sales should not have null values
"""

## Perform Date quality checks to ensure the order of transaction 
`sls_order_dt <<  sls_ship_dt << sls_due_dt` 

In [0]:
df.select("*").where(col("sls_order_dt") > col("sls_order_dt")).display()
df.select("*").where(col("sls_ship_dt") > col("sls_due_dt")).display()


## Check that all dates have a length of atleast 8 numbers, signifying yyyy mm dd

In [0]:
df.select("*").where(
    (F.length(col("sls_order_dt")) != 8) 
    |
    (F.length(col("sls_ship_dt")) != 8) 
    |
    (F.length(col("sls_due_dt")) != 8)
    
).display()

"""
    sls_order_dt has values with length lessthan 8 numbers
"""

## Check sales, quantity, and price for zero or negative values

In [0]:
# Check for zero or negative values in sls_sales (sales sales must be atleast 1 and never negative)
df.select("*").filter(col("sls_sales") <= 0).display()
"""
    Negative or zero values found in sls_sales
""" 

In [0]:
# Check for zero or negative values in sls_quantity (sales quantity must be atleast 1 and never negative)
df.select("*").filter(col("sls_quantity") <= 0).display()

In [0]:
# Check for zero or negative values in sls_price (sales price must be atleast 1 and never negative)
df.select("*").filter(col("sls_price") <= 0).display()

"""
    negative values found in sls_price
""" 

In [0]:
(
    df.select(col("sls_price"), col("sls_quantity"), col("sls_sales"))
    .distinct()
    .where(
        (col("sls_sales") != (col("sls_price") * col("sls_quantity"))) 
        | (col("sls_sales").isNull()) 
        | (col("sls_sales") <= 0)
        | (col("sls_price").isNull()) 
        | (col("sls_price") <= 0)
        | (col("sls_quantity").isNull()) 
        | (col("sls_quantity") <= 0)
    )
    .orderBy("sls_price", "sls_quantity", "sls_sales")
).display()


"""
    According to the business rule, sls_sales is the product of sls_quantity and sls_price
    sls_sales cannot be null 
    sls_sales must not be zero and below
"""

## Check if the sls_cust_id match customer_id on the crm_cust_info

In [0]:
query = """
    SELECT * FROM workspace.bronze_schema.crm_sales_details
    WHERE sls_cust_id NOT IN (SELECT customer_id FROM workspace.silver_schema.crm_cust_info)
"""
spark.sql(query).display()

# Data notes
- From printschema, it is deduced that the dates are of type IntergerType. Should be casted to DateType
- No white spaces detected in the string columns
- Dates seem correct, Order date < Shipping date < Due date
- sls_order_dt has date where character length not equal to 8 . Should be replace with null
- Negative or zero or null values found in sls_price and sls_sales column
- data has rows where product of sls_price and sls_quantity is not equal to sls_sales


# Transform the data

## Triming the StringType columns

In [0]:
for field in df.schema.fields:
    column = field.name
    column_type = field.dataType

    if isinstance(column_type, StringType):
        print(f"Trimming {column_type} column {column} ...")
        df = df.withColumn(
            column,
            F.trim(col(column))
        )
        print(f"Column {column} trimming completed! \n")

## Type casting the the date columns

In [0]:
df.display()

In [0]:
# Create a list of all of all date columns as writen in the df
# Because date is IntergerType, we cant cast directly to DateType. We have to convert to StringType first.

date_columns = ["sls_order_dt", "sls_ship_dt", "sls_due_dt"]

for column in date_columns:
    """
        loop through the date column list and perform int > string > date type conversion.
        Dates that have characters length not equal to  8  are replaced with None.
        We use the pyspark when/otherwise function and to_date to cast to date 
    """
    df = df.withColumn(
        column,
        F.when(F.length(col(column).cast("string")) != 8, None)
        .otherwise(
            F.to_date(col(column).cast("string"), "yyyyMMdd")
        )
    )

df.display()

# The date column has been fixed. column is now DateType

## Cleaning the price columns

In [0]:
df = (
    df.withColumn(
        "sls_sales",
        F.when(
            (col("sls_sales").isNull())
            | (col("sls_sales") <= 0)
            | (col("sls_sales") != (col("sls_quantity") * col("sls_price"))),
            (col("sls_quantity") * F.abs(col("sls_price")))

        ).otherwise(
            col("sls_sales")
        )

    ).withColumn(
        "sls_price",
        col("sls_sales") / col("sls_quantity")
    )
)

In [0]:
df.display()

# Write the data into the silver schema

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver_schema.crm_sales_details")

# Read crm_sales_details from silver table

In [0]:
crm_sales_details_df = spark.table("workspace.silver_schema.crm_sales_details")
crm_sales_details_df.display()